# 🧠 SearchMind API — Interactive Google Colab Demo

Welcome to the interactive Google Colab notebook for **SearchMind API**, the production-grade AI-native web search & retrieval platform for LLM agents, LangGraph, and RAG pipelines.

### In this notebook, you will test:
1. ⚡ **Intelligent Search (`/v1/search`)**: Multi-provider search with fallback & LLM answer synthesis.
2. 📄 **Clean Content Extraction (`/v1/extract`)**: Strips ads/boilerplate with optional Playwright JS rendering.
3. 🔬 **Deep Multi-Angle Research (`/v1/research`)**: Dynamic query expansion & multi-source synthesis.
4. 🤖 **LangGraph ReAct Agent Integration**: Seamless agentic tool use.
5. 📊 **Usage & Quota Monitoring (`/v1/usage`)**: Real-time quota tracking.

## 📦 Step 1: Install Dependencies & SDK

In [ ]:
# Clone repo and install SDK
!git clone https://github.com/pritpatel2412/SearchMind-API.git
%cd SearchMind-API/sdk
!pip install -q -e . httpx pydantic langchain langchain-openai langgraph
%cd ../..

## 🔑 Step 2: Configure API Key & Endpoint

Get your API key from [https://searchmind-api.vercel.app](https://searchmind-api.vercel.app) or use your deployed backend.

In [ ]:
import os
import getpass

# Enter your SearchMind API Key (starts with sm_live_...)
SEARCHMIND_API_KEY = getpass.getpass("Enter your SearchMind API Key (sm_live_...): ")
os.environ["SEARCHMIND_API_KEY"] = SEARCHMIND_API_KEY

# Set Base URL (default is live Render deployment)
BASE_URL = "https://searchmind-api.onrender.com/v1"
os.environ["SEARCHMIND_BASE_URL"] = BASE_URL

print(f"✅ Configured SearchMind API Endpoint: {BASE_URL}")

## 🔍 Step 3: Test Intelligent Web Search

In [ ]:
import sys
sys.path.append("SearchMind-API/sdk")
from searchmind import SearchMindClient

client = SearchMindClient(api_key=SEARCHMIND_API_KEY, base_url=BASE_URL)

query = "Latest breakthrough in autonomous LLM agents 2026"
print(f"Executing search for: '{query}'...\n")

response = client.search(
    query=query,
    num_results=4,
    search_depth="basic",
    include_answer=True
)

print("=" * 60)
print("🧠 AI SYNTHESIZED ANSWER:")
print("=" * 60)
print(response.answer)
print("\n" + "=" * 60)
print("📚 TOP SOURCES & CITATIONS:")
print("=" * 60)
for idx, res in enumerate(response.results, 1):
    print(f"[{idx}] {res.title}")
    print(f"    URL: {res.url}")
    print(f"    Relevance Score: {res.score}")
    print(f"    Snippet: {res.content[:150]}...\n")

## 📄 Step 4: Test Clean Content Extraction (`/v1/extract`)

In [ ]:
urls_to_extract = [
    "https://docs.python.org/3/whatsnew/3.12.html",
    "https://fastapi.tiangolo.com/"
]

print(f"Extracting clean content from: {urls_to_extract}...\n")
extract_res = client.extract(urls=urls_to_extract, max_content_length=1500)

print(f"Extracted: {extract_res.extracted_count} success | {extract_res.failed_count} failed\n")
for page in extract_res.results:
    print(f"🔹 Title: {page.title}")
    print(f"   URL: {page.url}")
    print(f"   Word Count: {page.word_count}")
    print(f"   Method: {page.extraction_method}")
    print("   Sanitized Markdown Snippet:")
    print(f"   {page.content[:300]}...\n" + "-"*40)

## 🔬 Step 5: Test Deep Multi-Angle Research (`/v1/research`)

*(Requires Pro / Enterprise Plan)*

In [ ]:
try:
    research_query = "Compare vector databases vs relational databases for RAG in 2026"
    print(f"Running deep research for: '{research_query}'...\n")
    
    research_res = client.research(query=research_query, max_sources=5, include_summary=True)
    
    print("=" * 60)
    print("📑 COMPREHENSIVE RESEARCH SUMMARY:")
    print("=" * 60)
    print(research_res.summary)
    print("\nSources Analyzed:")
    for s in research_res.sources:
        print(f"- {s.title} ({s.url})")
except Exception as e:
    print(f"Note: Deep Research is an Enterprise tier endpoint: {e}")

## 🤖 Step 6: Test LangGraph / LangChain Agent with SearchMind Tool

In [ ]:
import os

# Optional: Set OpenAI API Key if testing LangGraph
openai_key = getpass.getpass("Enter your OPENAI_API_KEY (optional, press Enter to skip): ")

if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    from langchain_openai import ChatOpenAI
    from langgraph.prebuilt import create_react_agent
    from searchmind.langgraph_tool import create_searchmind_tools

    tools = create_searchmind_tools(client)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(llm, tools)

    agent_prompt = "What is the newest version of Python and what are its key features? Use SearchMind to find out."
    print(f"Asking Agent: '{agent_prompt}'...\n")
    
    response = agent.invoke({"messages": [("user", agent_prompt)]})
    print("\n🤖 Agent Final Answer:")
    print(response["messages"][-1].content)
else:
    print("Skipped LangGraph agent execution (no OpenAI key provided).")

## 📊 Step 7: Check API Quota & Usage Statistics

In [ ]:
usage = client.get_usage()
print("=" * 40)
print("📊 YOUR SEARCHMIND USAGE STATS")
print("=" * 40)
print(f"Billing Period:   {usage.period}")
print(f"Total Requests:   {usage.total_requests} / {usage.monthly_limit}")
print(f"Quota Remaining:  {usage.remaining_requests} requests")
print(f"Percentage Used:  {usage.percentage_used}%")
print("=" * 40)